# AI Verilog/SystemVerilog RTL Generator & Verification Agent

This notebook adapts the original Gemini + LangChain agent structure into an AI agent for hardware design.

Workflow:

Natural-language / Python algorithm specification → RTL generation → Testbench generation → Compile → Simulate → Analyze failures → Repair → Re-test.

The agent generates hardware RTL from the intended behavior of the supplied Python/algorithm. It does not blindly translate arbitrary Python syntax into synthesizable hardware.


## 1. Install dependencies

In [ ]:
!pip install -qU langchain-google-genai langchain-core langchain


## 2. Configure Gemini API

In [ ]:
import os
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("Gemini API key configured.")
except Exception as e:
    print("Add GEMINI_API_KEY to Colab Secrets first.")
    raise e


## 3. Initialize the model

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)


## 4. Install the simulator

Icarus Verilog is used because it is simple to automate in Colab.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y iverilog


## 5. Define the RTL tools

In [ ]:
import os
import subprocess
from langchain_core.tools import tool

WORKDIR = "/content/rtl_agent"
os.makedirs(WORKDIR, exist_ok=True)

@tool
def write_file(filename: str, content: str) -> str:
    """Write a Verilog/SystemVerilog source file into the RTL workspace."""
    if not filename.endswith((".v", ".sv", ".vh", ".svh")):
        return "ERROR: only Verilog/SystemVerilog files are allowed."
    path = os.path.join(WORKDIR, os.path.basename(filename))
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Wrote {path}"

@tool
def read_file(filename: str) -> str:
    """Read a Verilog/SystemVerilog file from the RTL workspace."""
    path = os.path.join(WORKDIR, os.path.basename(filename))
    if not os.path.exists(path):
        return f"ERROR: {filename} does not exist."
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

@tool
def compile_verilog(top_file: str, testbench_file: str) -> str:
    """Compile RTL and its testbench with Icarus Verilog."""
    rtl = os.path.join(WORKDIR, os.path.basename(top_file))
    tb = os.path.join(WORKDIR, os.path.basename(testbench_file))
    out = os.path.join(WORKDIR, "sim.out")
    p = subprocess.run(
        ["iverilog", "-g2012", "-o", out, rtl, tb],
        capture_output=True, text=True
    )
    return f"EXIT_CODE={p.returncode}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}"

@tool
def run_simulation() -> str:
    """Run the compiled Icarus Verilog simulation."""
    out = os.path.join(WORKDIR, "sim.out")
    if not os.path.exists(out):
        return "ERROR: simulation executable does not exist. Compile first."
    p = subprocess.run(
        ["vvp", out], capture_output=True, text=True, timeout=30
    )
    return f"EXIT_CODE={p.returncode}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}"

@tool
def list_workspace() -> str:
    """List files in the RTL workspace."""
    return "\n".join(sorted(os.listdir(WORKDIR)))


## 6. Create the Verilog engineering agent

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

tools = [write_file, read_file, compile_verilog, run_simulation, list_workspace]

SYSTEM_PROMPT = r"""
You are an expert RTL design and verification engineer.

Convert the user's hardware requirement or Python algorithm into synthesizable
Verilog/SystemVerilog and a self-checking testbench.

Rules:
1. Understand intended hardware behavior before writing RTL.
2. Do not blindly translate software-only Python constructs into hardware.
3. Prefer SystemVerilog (.sv).
4. Define clock, reset, inputs, outputs, widths, and timing.
5. Generate a self-checking testbench.
6. Include directed corner cases and random tests when useful.
7. Use assertions when appropriate.
8. Use nonblocking assignments for sequential logic.
9. Avoid accidental latches and multiple drivers.
10. Never claim verification passed without actually compiling and simulating.
11. If compilation or simulation fails, inspect the error, repair the RTL/TB,
    and test again.
12. Keep repair iterations bounded.

Preferred files:
- design.sv
- tb.sv

Final report:
- hardware interpretation
- generated files
- compile result
- simulation result
- test/pass/fail counts when available
- remaining limitations
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)


## 7. Convert a Python algorithm/specification

In [ ]:
python_algorithm = r"""
def add(a, b):
    return a + b
"""

request = f"""
Convert the following Python algorithm/intended behavior into synthesizable
SystemVerilog hardware.

PYTHON / ALGORITHM:
{python_algorithm}

Requirements:
- Create design.sv
- Create tb.sv
- Make the testbench self-checking
- Compile and simulate both
- If anything fails, repair it and re-run
- Do not claim PASS unless the simulator actually passes
"""

result = agent.invoke({
    "messages": [HumanMessage(content=request)]
})

for m in result["messages"]:
    if hasattr(m, "pretty_print"):
        m.pretty_print()


## 8. Inspect generated files

In [ ]:
print(list_workspace.invoke({}))

print("===== design.sv =====")
print(read_file.invoke({"filename": "design.sv"}))

print("===== tb.sv =====")
print(read_file.invoke({"filename": "tb.sv"}))


## 9. Manual compile/simulation check

In [ ]:
print(compile_verilog.invoke({
    "top_file": "design.sv",
    "testbench_file": "tb.sv"
}))

print(run_simulation.invoke({}))


## 10. Download generated RTL and testbench

In [ ]:
from google.colab import files

for name in ["design.sv", "tb.sv"]:
    path = os.path.join(WORKDIR, name)
    if os.path.exists(path):
        files.download(path)


## 11. Next upgrades

- Python-to-hardware pattern library
- SystemVerilog Assertions (SVA)
- Reference models / scoreboards
- Random and constrained-random testing
- Coverage
- Waveform generation and analysis
- Verilator
- Yosys synthesis checks
- Linting
- RAG over Verilog/SystemVerilog documentation
- Separate Architect / RTL / Verification / Debug agents
- Structured JSON outputs
- Git versioning for every generated revision

Core principle:

**generate → compile → simulate → inspect → repair → verify**
